In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/processed/recalls.duckdb")
def q(sql):
    return con.execute(sql).df()

sql = open("../sql/02_panel.sql").read()
for stmt in [s for s in sql.split(";") if s.strip()]:
    con.execute(stmt)

print(q("SELECT COUNT(*) AS rows FROM panel").to_string(index=False))

   rows
2363007


In [2]:
print(q("""
SELECT COUNT(*)                                        AS rows,
       SUM(target)                                     AS positive_rows,
       ROUND(100.0 * AVG(target), 3)                   AS pct_positive,
       COUNT(DISTINCT campaign) FILTER (WHERE target = 1) AS recalls_represented
FROM panel
""").to_string(index=False))

print("\nBy year:")
print(q("""
SELECT YEAR(month) AS yr, COUNT(*) AS rows, SUM(target) AS positives,
       ROUND(100.0 * AVG(target), 3) AS pct
FROM panel GROUP BY 1 ORDER BY 1
""").to_string(index=False))

print("\nRecalls represented, by initiator:")
print(q("""
SELECT initiated_by, COUNT(DISTINCT campaign) AS recalls
FROM panel WHERE target = 1 GROUP BY 1 ORDER BY 2 DESC
""").to_string(index=False))

   rows  positive_rows  pct_positive  recalls_represented
2363007        20681.0         0.875                 1667

By year:
  yr   rows  positives   pct
2016 265613     2202.0 0.829
2017 248523     1885.0 0.758
2018 249964     1916.0 0.767
2019 254721     1807.0 0.709
2020 230182     1901.0 0.826
2021 210333     1951.0 0.928
2022 185975     1620.0 0.871
2023 206582     2040.0 0.988
2024 213399     2071.0 0.970
2025 219825     2422.0 1.102
2026  77890      866.0 1.112

Recalls represented, by initiator:
initiated_by  recalls
         MFR     1393
         ODI      257
        OVSC       17


In [3]:
con.close()